# Prompt engineering: comparing variants

This notebook iterates on a prompt by evaluating several variants against the same dataset.

## The loop

Prompt engineering here means: pick a prompt, run it against a dataset, read the scores and the reasoning, change the prompt, run again.
The dataset and the graders stay fixed; only the prompt in `run_prompt` moves.
This notebook factors the loop out: each variant lives in its own string, and a single cell runs the evaluation across all of them.

## Pipeline shape

1. **Dataset generation** (`evaluator.generate_dataset`): one call generates a list of diverse `ideas`, then one call per idea produces a full test case with `prompt_inputs` and `solution_criteria`.
   The result is saved to `dataset.json`.
   Run this once per prompt_eng session; keep the dataset fixed across variants for comparability.
2. **Evaluation per variant** (`evaluator.run_evaluation`): for each test case, `run_prompt(prompt_inputs)` emits a solution and `grade_output` scores it with a sonnet-based grader.
   `extra_criteria` adds mandatory requirements that score ≤ 3 if violated.
   Per-variant results go to `output_<variant>.json` / `output_<variant>.html`.
   The library's progress and per-variant score prints are silenced inside the loop; the final cell prints only the summary.

## Model split

- **Dataset generator** (`generate_unique_ideas`, `generate_test_case`): `claude-haiku-4-5` (default).
- **Solver** (`run_prompt`): `claude-haiku-4-5` (default).
  This is the model under test.
- **Grader** (`grade_output`): `claude-sonnet-4-5` (explicit override).

## Concurrency

`PromptEvaluator` uses a `ThreadPoolExecutor` with `max_concurrent_tasks` workers for both dataset generation and evaluation.
16 is a safe default for the Anthropic API; raise it only if you have higher rate limits.


In [1]:
import contextlib
import io
import textwrap
from statistics import mean

from evals import PromptEvaluator

from chat import add_user_message, chat

evaluator = PromptEvaluator(max_concurrent_tasks=16)

In [2]:
# Run once per session. The variants below are evaluated against this fixed dataset.
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="dataset.json",
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [3]:
def block(text):
    return textwrap.dedent(text).strip("\n")


input_spec_block = block("""
    - Height: {height}
    - Weight: {weight}
    - Goal: {goal}
    - Dietary restrictions: {restrictions}
""")

qualities_block = block("""
    The output must have the following qualities:
    - An accurate daily calorie amount tailored to the athlete.
    - A macronutrient breakdown with explicit grams of protein, fat, and carbs.
    - A meal schedule with a time for each meal.
    - Foods that satisfy the dietary restrictions.
    - Portion sizes listed in grams.
""")

steps_block = block("""
    Follow these steps internally, then output only the final meal plan:
    1. Estimate the athlete's daily calorie needs from height, weight, and goal.
    2. Choose a macronutrient split (grams of protein, fat, and carbs) appropriate for the goal.
    3. Design 4 to 6 meals distributed through the day, each with a specific time.
    4. For every meal, pick foods that fit the dietary restrictions and give portion sizes in grams.
    5. Keep the meals budget-friendly if a budget is mentioned.
    6. Verify that the sum of meals matches the calorie and macro targets from steps 1 and 2.
""")

sample_input_block = block("""
    <sample_input>
    height: 178
    weight: 85
    goal: strength and power development for weightlifting
    restrictions: vegetarian, nut allergy
    </sample_input>
""")

ideal_output_block = block("""
    <ideal_output>
    # One-Day Meal Plan for Vegetarian Weightlifter

    **Daily Targets:**
    - Calories: 2,800
    - Protein: 140g | Carbs: 350g | Fat: 78g

    ---

    ## Meal 1: Breakfast (7:00 AM)
    **Oatmeal Power Bowl** | 650 cal | 18g P | 95g C | 18g F
    - Rolled oats: 80g
    - Whole milk: 250ml
    - Banana: 120g
    - Honey: 20g
    - Sunflower seeds: 15g
    - Cinnamon: 2g

    ---

    ## Meal 2: Mid-Morning Snack (10:00 AM)
    **Greek Yogurt with Berries** | 280 cal | 20g P | 38g C | 4g F
    - Plain Greek yogurt (0%): 200g
    - Mixed berries (frozen): 100g
    - Granola (nut-free): 30g
    - Honey: 10g

    ---

    ## Meal 3: Lunch (1:00 PM)
    **Lentil & Quinoa Bowl** | 620 cal | 28g P | 85g C | 15g F
    - Cooked quinoa: 150g
    - Red lentils (cooked): 120g
    - Spinach: 80g
    - Olive oil: 12g
    - Feta cheese: 40g
    - Chickpeas (canned): 80g
    - Lemon juice & herbs: 5g

    ---

    ## Meal 4: Pre-Workout (3:30 PM)
    **Rice Cakes with Protein** | 380 cal | 18g P | 65g C | 6g F
    - White rice cakes: 80g
    - Hummus (sesame): 40g
    - Banana: 100g
    - Hemp seeds: 15g

    ---

    ## Meal 5: Dinner (6:30 PM)
    **Paneer Stir-Fry with Rice** | 650 cal | 32g P | 65g C | 18g F
    - Paneer cheese: 150g
    - Brown rice (cooked): 180g
    - Mixed vegetables (broccoli, peppers, carrots): 200g
    - Coconut oil: 15g
    - Garlic & ginger: 5g
    - Low-sodium soy sauce: 10g

    ---

    ## Meal 6: Evening Snack (8:30 PM)
    **Cottage Cheese & Fruit** | 220 cal | 24g P | 22g C | 3g F
    - Cottage cheese (low-fat): 200g
    - Pineapple chunks: 80g
    - Pumpkin seeds: 10g

    ---

    ## **Daily Totals**
    - **Calories: 2,800** ✓
    - **Protein: 140g** ✓
    - **Carbs: 370g** ✓
    - **Fat: 64g** ✓

    *All meals are vegetarian and nut-free. This plan prioritizes protein timing around workouts and provides adequate carbohydrates for strength/power development.*
    </ideal_output>
    The solution comprehensively meets all mandatory requirements by including daily caloric totals,
    macronutrient breakdowns, and detailed meals with exact foods, portions, and timing.
    It also satisfies all secondary criteria: covers a full day with all meal types,
    provides adequate protein from appropriate vegetarian sources (140g exceeds the 85-100g minimum),
    accounts for high caloric needs of a strength athlete,
    and contains no nuts or nut-based products.
    The only issues are minor numerical inconsistencies between the header targets and final totals,
    which do not materially impact the plan's usability or compliance with requirements.
""")

example_block = "\n".join(
    [
        "Here is a sample input paired with an ideal output:",
        "<example>",
        sample_input_block,
        ideal_output_block,
        "</example>",
    ]
)

# Prompt variants. Each is a template string with the keys declared in `prompt_inputs_spec`.
# Add new variants here as you iterate.

# 1. Simple
task_statement_1 = "What should this person eat?"
prompt_1_simple = task_statement_1 + "\n\n" + input_spec_block

# 2. Clear and direct
task_statement_2 = "Generate a one-day meal plan that meets the dietary restrictions for the athlete below."
prompt_2_clear_and_direct = task_statement_2 + "\n\n" + input_spec_block

# 3a. Be specific by listing qualities the output must have.
prompt_3a_list_qualities = prompt_2_clear_and_direct + "\n\n" + qualities_block

# 3b. Be specific by walking the model through the steps to produce the output.
prompt_3b_provide_steps = prompt_2_clear_and_direct + "\n\n" + steps_block

# 4. Add examples.
prompt_4_add_examples = prompt_3b_provide_steps + "\n\n" + example_block

variants = {
    "1_simple": prompt_1_simple,
    "2_clear_and_direct": prompt_2_clear_and_direct,
    "3a_list_qualities": prompt_3a_list_qualities,
    "3b_provide_steps": prompt_3b_provide_steps,
    "4_add_examples": prompt_4_add_examples,
}

In [4]:
# Factory: takes a prompt template, returns a `run_prompt(prompt_inputs)` function
# that `evaluator.run_evaluation` can call for each test case.
def make_run_prompt(template):
    def run_prompt(prompt_inputs):
        messages = []
        add_user_message(messages, template.format(**prompt_inputs))
        return chat(messages)

    return run_prompt

In [6]:
extra_criteria = block("""
    The output should include:
    - Daily caloric total.
    - Macronutrient breakdown.
    - Meals with exact foods, portions, and timing.
""")

scores = {}
for name, template in variants.items():
    with contextlib.redirect_stdout(io.StringIO()):
        results = evaluator.run_evaluation(
            run_prompt_function=make_run_prompt(template),
            dataset_file="dataset.json",
            extra_criteria=extra_criteria,
            json_output_file=f"output_{name}.json",
            html_output_file=f"output_{name}.html",
        )
    scores[name] = mean(r["score"] for r in results)

for name, score in scores.items():
    print(f"  {name}: {score:.2f}")

  1_simple: 2.00
  2_clear_and_direct: 7.33
  3a_list_qualities: 6.67
  3b_provide_steps: 7.00
  4_add_examples: 8.00
